# T1 · The harmonic oscillator

Run from top to bottom with NumPy, PyTorch, Matplotlib and a Jupyter kernel. The reference trajectory is available to the evaluator. Only the sampled context is an observation.

## 1. From forces to motion
A spring pulls a mass back toward equilibrium. Doubling the displacement doubles the opposing force: $m\ddot u=-ku$. With $\omega=\sqrt{k/m}$, this is $\ddot u=-\omega^2u$. There is no friction or forcing.

## 2. What fixes one realization?
Amplitude sets the excursion, angular frequency sets the clock, and phase locates the mass on its cycle at time zero. Position alone does not specify the direction of travel. One draw of $(A,\omega,\phi)$ fixes both $u=A\cos(\omega t+\phi)$ and $\dot u=-A\omega\sin(\omega t+\phi)$. ID uses independent uniform draws on $[0.5,1.5]$, $[0.5,1.5]$ and $[0,2\pi)$ respectively. Only the frequency range changes for OOD: $[1.8,2.5]$.

In [ ]:
from pathlib import Path
import sys
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'worldlab/datasets/oscillator.py').exists())
sys.path.insert(0, str(root))
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from worldlab.datasets.oscillator import OscillatorDataset as Dataset, make_realization
from worldlab.datasets._dynamics_common import N_OBS, DT_REF
position, velocity = 'u', 'u_t'
ds = Dataset(split='train')
r = make_realization(seed=ds.seeds[0])
repeat = make_realization(seed=r.seed)
assert r.params == repeat.params and all(getattr(r,k).tobytes() == getattr(repeat,k).tobytes() for k in ('t_ref','z','z_t'))
print('seed', r.seed, 'parameters', r.params, 'reference shape/dtype', r.z.shape, r.z.dtype)
print('views per realization:', N_OBS, 'reference spacing (s):', DT_REF)

## 3. Validate the simulator
Evaluate the closed form directly at every requested time; no solver step or interpolation is needed. Per unit mass, kinetic plus spring energy must equal the same constant at every point. The flat physical energy is plotted without a misleading offset axis.

In [ ]:
a,w,phi = (r.params[k] for k in ('A','omega','phi'))
E = (r.z_t**2 + (w*r.z)**2)/2
err = np.max(abs(E - (a*w)**2/2))
assert err < 1e-9
print('maximum absolute energy error:', err)
fig, ax = plt.subplots(figsize=(7,2.3))
ax.plot(r.t_ref,E); ax.set(xlabel='Time (s)',ylabel='Energy per unit mass',ylim=(0,1.2*E[0]))
plt.show()

## 4–5. What is observed, and what is queried?
Each realization has 32 independently seeded configurations per mode. A sensor records position, velocity, or both, without noise. The step is chosen from 0.05, 0.1, 0.2 and 0.5 seconds; the cutoff is uniform between 60% and 90% of the duration. The last sampled time (dashed line) is the hull boundary, usually **before** the cutoff (dotted line).

Each row uses its own real context, query and sensor channels; the first two share physical seed and configuration index. Mode enters the view seed, so changing mode can change the observations as well as the query. Parameter OOD changes the physical draw: here its query is inside its own hull.

In [ ]:
views = [ds[0], Dataset(split='train',mode='extrapolate')[0], Dataset(split='param_ood')[0]]
fig, axes = plt.subplots(3,2,figsize=(11,7),sharex=True)
for s, panels, color in zip(views,axes,['tab:green','tab:red','tab:purple']):
    sensor = s['sensor_type']
    channels = [sensor] if sensor in (position,velocity) else [position,velocity]
    assert s['t_max'] == s['t_i'][-1] < s['t_obs']
    assert s['d_q'] == max(0,s['t_q']-s['t_max'])
    if s['mode']=='extrapolate': assert s['t_q'] >= s['t_obs']
    else: assert 0 <= s['t_q'] <= s['t_max']
    for ax,ch,field in zip(panels,(position,velocity),('z','z_t')):
        ax.plot(s['t_ref'],s[field],color='0.7',label='Full truth')
        if ch in channels:
            j=channels.index(ch)
            ax.scatter(s['t_i'],s['values'][:,j],s=9,label='Observed')
            ax.scatter(s['t_q'],s['target'][j],marker='*',s=110,color=color,label='Query target')
        ax.axvline(s['t_max'],ls='--',color='k'); ax.axvline(s['t_obs'],ls=':',color='0.5')
        ax.set_ylabel(ch); ax.legend(fontsize=7)
    print(f"{s['mode']:12s} seed={s['seed']}  sensor={sensor}  t_max={s['t_max']:.3f}  "
          f"cutoff={s['t_obs']:.3f}  query={s['t_q']:.3f}  d_q={s['d_q']:.3f}")
assert views[0]['realization_id']==views[1]['realization_id']
for ax in axes[-1]: ax.set_xlabel('Time (s)')
fig.tight_layout(); plt.show()

T1 evaluates the analytic solution at the continuous query time. A positive distance is measured from the last observation, not from the window cutoff.

## 6. A real training batch
Inspect one item before batching. Pad time to the longest context and keep two named channels so that a velocity-only sensor is never mistaken for a position sensor. A Boolean mask distinguishes observed zeros from padding and missing channels. The reference trajectory is not an input to this batch.

In [ ]:
s=views[0]
print('Item keys:', ', '.join(s))
print('Arrays:', {k:(v.shape,str(v.dtype)) for k,v in s.items() if isinstance(v,np.ndarray)})
def collate(rows):
    lengths = torch.tensor([len(r['t_i']) for r in rows])
    B, L = len(rows), int(lengths.max())
    context = torch.zeros(B,L,2,dtype=torch.float64)
    times = torch.zeros_like(context[:,:,0])
    observed = torch.zeros_like(context,dtype=torch.bool)
    target = torch.zeros_like(context[:,0])
    target_mask = torch.zeros_like(observed[:,0])
    for i,r in enumerate(rows):
        n = len(r['t_i'])
        times[i,:n] = torch.from_numpy(r['t_i'])
        channels = {position:[0],velocity:[1]}.get(r['sensor_type'],[0,1])
        for j,c in enumerate(channels):
            context[i,:n,c] = torch.from_numpy(r['values'][:,j])
            observed[i,:n,c] = True
            target[i,c] = float(r['target'][j])
            target_mask[i,c] = True
    assert torch.equal(observed.any(dim=2).sum(dim=1), lengths)
    return dict(context=context,times=times,observed=observed,lengths=lengths,
                query=torch.tensor([r['t_q'] for r in rows],dtype=torch.float64),
                target=target,target_mask=target_mask)
batch = next(iter(DataLoader(ds,batch_size=4,shuffle=False,collate_fn=collate)))
print('Batch:', {k:tuple(v.shape) for k,v in batch.items()})
print('Unpadded lengths:', batch['lengths'].tolist())
assert batch['lengths'].unique().numel()>1

## What this testbed tests in NO-JEPA

T1 is the minimal viable experiment: the evaluator knows the exact trajectory, so target error cannot be blamed on an ODE solver. Comparing interpolation with errors at increasing distance beyond the hull tests whether the representation carries phase and frequency into an unobserved future.

The frequency-held-out split asks a different question: can the same prediction rule work on a clock outside the training range? Keeping observation laws fixed and querying inside the OOD hull separates this parameter shift from the difficulty of a longer prediction horizon.